# noise_reachability step 15: are 6/14 and 6/20 unreachable, and did distillation lose 7/38's solutions? (Kaggle, 2 x T4)

Step 15 of `noise_reachability/README.md`. No training. *Test A:* the base with 400 fresh `repeat` noises on each of 6/14
and 6/20, every success replayed once. *Test B:* the base and step 10's N7_0-N7_2 on the same 200 fresh `repeat` noises
on 7/38.

**Needs one input:** Add Input > Your Work > the step 10 notebook, its **last** version (adapters N7_0-N7_2; their sha256
are checked). **Run with Save & Run All (Accelerator: GPU T4 x2, Internet on).** Nothing needs editing. The wrapper
check, the scene checks and the harness run first; a failure stops the run.

**This run needs two sessions (about 11 h and 1.5 h, an estimate).** When step 14 says work remains, add this version's output as input
as well (**Add Input > Your Work**, only the latest step 15 version; keep step 10's), and Save & Run All again. Step 6
must then print `RESUMING`.

Download from the Output tab: `step15_results.zip`.

## 1. Settings

In [ ]:
import csv, glob, hashlib, json, os, re, shutil, signal, subprocess, sys, time
from pathlib import Path

T_SESSION = time.time()
REPO_URL = "https://github.com/swagatk/efficient_vla.git"
REPO_REF = "16a03fb31a21e4ed96c308ed99585317082bbaf2"
assert REPO_REF != "SET_AFTER_COMMIT", "this notebook has no reviewed commit yet"
REPO = "/tmp/efficient_vla"        # code and venv live outside /kaggle/working, so they are not saved as output
VENV = "/tmp/venv"
PY = f"{VENV}/bin/python"

# The protocol of README.md step 15; nothing here is meant to be edited. The analysis refuses anything else.
CHECKPOINT_REPO, CHECKPOINT_REVISION = "lerobot/smolvla_libero", "a24f1f380461b665158d52acd140152c79e8b36d"
TASKS = [6, 7]                    # the wrapper check runs on both
N_ACTION_STEPS = 10
TASK_A, STATES_A = 6, [14, 20]        # test A: never solved by any noise
A_PARTS = ['9000-9199', '9200-9399']   # fresh repeat ids, two jobs per state
TASK_B, STATE_B = 7, 38                # test B: the base against N7_0-N7_2 on the same noises
B_IDS = "9000-9199"
B_POLICIES = ['base', 'N0', 'N1', 'N2']
SCENE_STATES = {6: (14, 20), 7: (38,)}
HARNESS_BASE_IDS = "6000-6002,6017,6019,6021,6023,6025,6030,6040"   # step 9's 7/38 episodes
HARNESS_N_STATE, HARNESS_N_SEEDS = 33, "16-19"   # step 10's N7_0 episodes
N7_SHA256 = {'N0': {'adapter_config.json': '33f536c16b513ab6673fa7106a9e38be80df456a246ccef0131f4d85aefdc91e', 'adapter_model.safetensors': '49c56ea1ea44cb6d4dd50c6362f4304531af410c355b03917c5cafa94bd00c40', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'N1': {'adapter_config.json': '903aae691044be234e4887dd43dce0c59962576684ce91f10451380009f5b7cd', 'adapter_model.safetensors': '497fa7e300c2a3650a3091016f92f6607400a0cdd13ad56f065e4efdb1347366', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}, 'N2': {'adapter_config.json': 'cc5ca0f30c9e85ef175afaa668532da3c7bcd92b60fa6e164ef1212a5e57b5c0', 'adapter_model.safetensors': '41a132f12b07fd7652b0d0b8692051eb1a8a4887e578552a832122aab3db7ea3', 'run_config.json': '550aad6fc781c7bf130daede67413757343cbe29c1b4ebab4ccbe7047c0f3f1a'}}
OUT_NAME = "step15"
OUT_ROOT = f"/kaggle/working/{OUT_NAME}"
STOP_AT_H = 11.0                   # no new work after this many hours of the session; running jobs are stopped
GB_PER_JOB = 6                     # base policy + one episode in RAM per simulator process

# Commits of the editable installs in the local lerobot_env that produced the existing results.
LEROBOT_COMMIT = "b6ec0060779550c0a157ae34feb89e0cf86012a8"
LIBERO_COMMIT = "8f1084e3132a39270c3a13ebe37270a43ece2a01"
print(f"step 15 at {REPO_REF}: test A {TASK_A}/{STATES_A} ids {A_PARTS}, test B {TASK_B}/{STATE_B} ids "
      f"{B_IDS} for {B_POLICIES}, at most 1614 episodes (14 harness, 800 test A, 800 "
      f"test B) plus replays, stop after {STOP_AT_H} h, output {OUT_ROOT}")


def sh(cmd, check=True, env=None):
    # Capture and print: notebooks do not show output that a subprocess writes directly.
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash", env=env, capture_output=True, text=True)
    print((r.stdout + r.stderr).rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"command failed with exit code {r.returncode}: {cmd}")
    return r


def hours_left():
    return STOP_AT_H - (time.time() - T_SESSION) / 3600

## 2. Hardware

In [ ]:
smi = sh("nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv,noheader")
gpus = [line.split(", ") for line in smi.stdout.strip().splitlines()]
assert gpus, "No GPU. Set Accelerator to GPU T4 x2."
for name, mem, cap, driver in gpus:
    assert float(cap) >= 7.5, f"{name} (compute capability {cap}) is not supported by the pinned PyTorch build; use GPU T4 x2"
CUDA_DRIVER = float(re.search(r"CUDA Version: ([0-9.]+)", sh("nvidia-smi").stdout).group(1))
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 1e6
avail_gb = int(re.search(r"MemAvailable:\s+(\d+)", open("/proc/meminfo").read()).group(1)) / 1e6
JOBS = max(1, min(2 * len(gpus), int((avail_gb - 4) // GB_PER_JOB), os.cpu_count()))
GPU_IDS = ",".join(str(i) for i in range(len(gpus)))
print(f"{len(gpus)} GPU(s), driver supports CUDA {CUDA_DRIVER}; {os.cpu_count()} CPU cores; "
      f"RAM {mem_gb:.0f} GB ({avail_gb:.0f} GB available) -> {JOBS} evaluations at a time on GPUs {GPU_IDS}")
assert len(gpus) == 2, "Set Accelerator to GPU T4 x2"
assert JOBS >= 4, f"RAM for {JOBS} SmolVLA processes; the notebook runs 4 at once"

## 3. Code

In [ ]:
if not Path(REPO, ".git").exists():
    sh(f"git clone {REPO_URL} {REPO}")
sh(f"git -C {REPO} fetch origin && git -C {REPO} checkout {REPO_REF} && git -C {REPO} pull --ff-only origin {REPO_REF}")
sh(f"git -C {REPO} log -1 --format='%H %s'")

## 4. Environment

The package versions of the local `lerobot_env` (as the earlier Kaggle notebooks). Takes several minutes.

In [ ]:
# Headless rendering: MuJoCo needs the NVIDIA EGL vendor file, which some images lack.
icd = Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}))
sh("apt-get install -y -qq libegl1 libgles2 libgl1 > /dev/null", check=False)

sh("python -m pip install -q uv")
if not Path(PY).exists():
    sh(f"uv venv {VENV} --python 3.12")

req = Path(REPO, "hybrid_diffusion", "requirements-colab.txt").read_text().splitlines()
if CUDA_DRIVER >= 13.0:
    TORCH_BUILD = "cu130 (as pinned)"
    reqs = req
else:
    cu = "cu128" if CUDA_DRIVER >= 12.8 else "cu126"
    assert CUDA_DRIVER >= 12.6, f"driver supports only CUDA {CUDA_DRIVER}; need 12.6 or newer"
    TORCH_BUILD = f"{cu} (driver too old for cu130)"
    torch_pins = [l for l in req if l.split("==")[0] in ("torch", "torchvision")]
    sh(f"uv pip install -q --python {PY} " + " ".join(f"'{p}+{cu}'" for p in torch_pins)
       + f" --index-url https://download.pytorch.org/whl/{cu} --extra-index-url https://pypi.org/simple")
    reqs = [l for l in req if not re.match(r"(torch|torchvision|triton|nvidia-)", l)]
print("PyTorch build:", TORCH_BUILD)
Path("/tmp/requirements.txt").write_text("\n".join(reqs) + "\n")
sh(f"uv pip install -q --python {PY} --no-deps -r /tmp/requirements.txt")
sh(f"uv pip install -q --python {PY} --no-deps 'lerobot @ git+https://github.com/huggingface/lerobot@{LEROBOT_COMMIT}'")

LIBERO = os.path.expanduser("~/LIBERO")
if not Path(LIBERO, ".git").exists():
    sh(f"git clone https://github.com/Lifelong-Robot-Learning/LIBERO.git {LIBERO}")
sh(f"git -C {LIBERO} checkout -q {LIBERO_COMMIT}")
Path(LIBERO, "libero", "__init__.py").touch()   # present in the local checkout, missing upstream
sh(f"uv pip install -q --python {PY} --no-deps -e {LIBERO}")

# LIBERO asks for these paths interactively on first import unless the config already exists.
root = f"{LIBERO}/libero/libero"
os.makedirs(os.path.expanduser("~/.libero"), exist_ok=True)
os.makedirs(f"{LIBERO}/libero/datasets", exist_ok=True)
Path(os.path.expanduser("~/.libero/config.yaml")).write_text(
    f"benchmark_root: {root}\nbddl_files: {root}/bddl_files\ninit_states: {root}/init_files\n"
    f"datasets: {LIBERO}/libero/datasets\nassets: {root}/assets\n"
)

## 5. Sanity checks, tests and the base checkpoint

In [ ]:
RUN_ENV = dict(os.environ, PATH=f"{VENV}/bin:" + os.environ["PATH"], MUJOCO_GL="egl", PYOPENGL_PLATFORM="egl",
               MPLBACKEND="Agg", WANDB_MODE="disabled", OMP_NUM_THREADS="1", PYTHONPATH=REPO,
               PYTHONFAULTHANDLER="1")
sh(f"""{PY} -c "
import torch, av
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], '| av', av.__version__)
" """, env=RUN_ENV)
egl = sh(f"{PY} -c \"import mujoco; c = mujoco.GLContext(64, 64); c.make_current(); print('EGL context ok')\"", check=False, env=RUN_ENV)
assert "EGL context ok" in egl.stdout, "EGL rendering failed"
sh(f"cd {REPO} && {PY} -m pytest tests/ -q -p no:cacheprovider", env=RUN_ENV)
r = sh(f"""{PY} -c "from huggingface_hub import snapshot_download; print(snapshot_download('{CHECKPOINT_REPO}', revision='{CHECKPOINT_REVISION}'))" """, env=RUN_ENV)
POLICY_PATH = r.stdout.strip().splitlines()[-1]
print("base checkpoint", POLICY_PATH)

## 6. Resume from an earlier version's output

In [ ]:
os.makedirs(OUT_ROOT, exist_ok=True)
previous = [p for p in sorted(glob.glob(f"/kaggle/input/**/{OUT_NAME}", recursive=True))
            if os.path.isdir(p) and os.path.exists(f"{p}/repo_commit.txt")]
assert len(previous) <= 1, f"attach only the latest version's output, found {previous}"
for prev in previous:
    print("RESUMING from", prev)
    shutil.copytree(prev, OUT_ROOT, dirs_exist_ok=True)
if not previous:
    print("FRESH START: no earlier output attached. If this is the second session, stop and add it as input.", flush=True)
head = sh(f"git -C {REPO} log -1 --format='%H'").stdout.strip()
first = Path(OUT_ROOT, "repo_commit.txt")
started_with = first.read_text().split()[0] if first.exists() and first.read_text().strip() else None
if started_with and started_with != head:
    raise RuntimeError(f"the run started at commit {started_with}, this version cloned {head}: set REPO_REF in "
                       f"step 1 to {started_with}")
with open(first, "a") as f:
    f.write(head + "\n")
for d in ("logs", "eval"):
    os.makedirs(f"{OUT_ROOT}/{d}", exist_ok=True)

## 7. Input: step 10's adapters N7_0-N7_2 (sha256 against step 10's archive manifest)

In [ ]:
found = [p for p in sorted(glob.glob("/kaggle/input/**/step10", recursive=True))
         if os.path.exists(f"{p}/runs/N{TASK_B}_0/adapter/run_config.json")]
assert len(found) == 1, f"attach step 10's last version as input (found {found})"
STEP10_IN = found[0]
print("step 10 output:", STEP10_IN, "| its commits:", open(f"{STEP10_IN}/repo_commit.txt").read().split())


def n_adapter(policy):
    return f"{STEP10_IN}/runs/N{TASK_B}_{policy[1:]}/adapter"


for policy, files in N7_SHA256.items():
    for name, want in files.items():
        got = hashlib.sha256(Path(n_adapter(policy), name).read_bytes()).hexdigest()
        assert got == want, f"{n_adapter(policy)}/{name}: sha256 {got}, expected step 10's {want}"
print(f"{len(N7_SHA256)} adapters: sha256 equal to step 10's archive manifest")

## 8. Helpers: parallel jobs with resume and a deadline

In [ ]:
def run_jobs(jobs, max_parallel, poll_s=30, log_every_s=600):
    """Run {name: (command, gpu)} with at most max_parallel at once; exit code 3 means 'call again' (more
    episodes in a fresh process). No job starts after the deadline and running ones are stopped at it.
    Returns {name: exit code}; a job stopped at the deadline gets None."""
    pending, running, codes, last_log = list(jobs), {}, {}, time.time()
    while pending or running:
        while pending and len(running) < max_parallel and hours_left() > 0:
            name = pending.pop(0)
            cmd, gpu = jobs[name]
            log = open(f"{OUT_ROOT}/logs/{name}.log", "a")
            env = dict(RUN_ENV, CUDA_VISIBLE_DEVICES=str(gpu))
            running[name] = (subprocess.Popen(cmd, shell=True, executable="/bin/bash", env=env, stdout=log,
                                              stderr=subprocess.STDOUT, start_new_session=True), log)
            time.sleep(5)  # staggered EGL start-up (FAIL-006)
        if hours_left() <= 0 and running:
            print(time.strftime("%H:%M"), "deadline: stopping", list(running), flush=True)
            for name, (proc, log) in running.items():
                if proc.poll() is None:
                    os.killpg(proc.pid, signal.SIGTERM)
            time.sleep(20)
            for name, (proc, log) in running.items():
                log.close()
                codes[name] = None
            for name in pending:
                codes[name] = None
            return codes
        if hours_left() <= 0 and not running:
            for name in pending:
                codes[name] = None
            return codes
        for name, (proc, log) in list(running.items()):
            if proc.poll() is None:
                continue
            log.close()
            del running[name]
            if proc.returncode == 3:
                pending.insert(0, name)
                continue
            codes[name] = proc.returncode
            print(time.strftime("%H:%M"), name, "exit", proc.returncode, flush=True)
        if time.time() - last_log > log_every_s:
            last_log = time.time()
            print(time.strftime("%H:%M"), f"{hours_left():.1f} h left; running {list(running)}; "
                  f"{len(pending)} waiting", flush=True)
        time.sleep(poll_s)
    return codes


FIELDS = ["task_id", "seed", "init_state", "success", "steps"]


def read_csv(path):
    return list(csv.DictReader(open(path))) if os.path.exists(path) else []


def write_csv(rows, out):
    os.makedirs(os.path.dirname(out), exist_ok=True)
    with open(out, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=FIELDS)
        w.writeheader()
        w.writerows({k: r[k] for k in FIELDS} for r in rows)
    return rows


def n_range(spec):
    a, b = (spec.split("-") + [spec])[:2]
    return int(b) - int(a) + 1


def failed(codes):
    return {k: c for k, c in codes.items() if c not in (0, None)}


FAILURE = None


def fail(message):
    """Keep going to the zip in step 15, which raises this."""
    global FAILURE
    FAILURE = message
    print("FAILED:", message, flush=True)






SEARCH = f"cd {REPO} && {PY} -u noise_reachability/deep_noise_search.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS} --family repeat"
SCENE = f"cd {REPO} && {PY} -u noise_reachability/check_scene.py --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}"
ANALYZE = f"cd {REPO} && {PY} -S noise_reachability/analyze_step15.py --root {OUT_ROOT}"
MAIN = f"{REPO}/self_filtered_bc/self_filtered_bc.py"


def task_dir(task):
    return f"{OUT_ROOT}/task_{task}"


def search_cmd(task, state, ids, out, policy="base"):
    lora = "" if policy == "base" else f" --lora_dir {n_adapter(policy)}"
    return f"{SEARCH} --task_id {task} --init_state {state} --noise_ids {ids} --output_dir {out}{lora}"


def harness_n_cmd():
    return (f"cd {REPO} && {PY} -u {MAIN} --task_id {TASK_B} --n_action_steps {N_ACTION_STEPS} --policy_path {POLICY_PATH} "
            f"--stage eval --eval_policies lora --lora_dir {n_adapter('N0')} --eval_seeds {HARNESS_N_SEEDS} "
            f"--eval_init_states {HARNESS_N_STATE} --eval_output_dir {OUT_ROOT}/harness/N0 --max_episodes_per_process 10")


def n_ids(spec):
    return sum(int(p.split("-")[1]) - int(p.split("-")[0]) + 1 if "-" in p else 1 for p in spec.split(","))


def run_stage(name, jobs, total, max_parallel=4):
    """True when the stage has no work left: a job that failed stops the run, the deadline leaves work for a next version."""
    print(f"{name}: {len(jobs)} of {total} jobs to run; {hours_left():.1f} h left", flush=True)
    if not jobs:
        return True
    if hours_left() <= 0:
        return False
    codes = run_jobs(jobs, max_parallel=max_parallel)
    if failed(codes):
        fail(f"{name} failed: {failed(codes)}")
        return False
    return all(c == 0 for c in codes.values()) and len(codes) == len(jobs)


def run_gate(which):
    if sh(f"{ANALYZE} --gate {which}", check=False, env=RUN_ENV).returncode != 0:
        fail(f"the {which} gate failed: investigate before spending more")
        return False
    return True


def scene_checked(task, state):
    """One state's scene check, in its own process, one at a time."""
    if hours_left() <= 0:
        return False
    r = sh(f"{SCENE} --task_id {task} --init_state {state} --output_dir {task_dir(task)}/scene", check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"the scene check failed to run on task {task} state {state}")
        return False
    return True


def scene_passed(task, state):
    p = Path(task_dir(task), "scene", f"scene_state{state}.json")
    return p.exists() and bool(json.loads(p.read_text())["same"])

## 9. Wrapper check on each task's real model (a few minutes); stops the run if it fails

In [ ]:
CHECK_DONE = True
for t in TASKS:
    r = sh(f"cd {REPO} && {PY} -u noise_reachability/check_step5a.py --task_id {t} --policy_path {POLICY_PATH} --n_action_steps {N_ACTION_STEPS}",
           check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"check_step5a failed on task {t}")
        CHECK_DONE = False

## 10. Scene checks of 6/14, 6/20 and 7/38

A state that fails is dropped, not replaced: it is written to `scene_state<i>.json`, and the report lists it.

In [ ]:
SCENES_DONE = CHECK_DONE and not FAILURE
for t, states in SCENE_STATES.items():
    for s in states:
        if SCENES_DONE and not scene_checked(t, s):
            SCENES_DONE = False
print({f"{t}/{s}": scene_passed(t, s) for t, states in SCENE_STATES.items() for s in states})

## 11. Harness: the base on 10 of step 9's 7/38 episodes, N7_0 on 4 of step 10's 7/33 episodes; stops the run if it fails

At least 13 of the 14 episodes must reproduce the committed rows exactly (success and number of steps).

In [ ]:
HARNESS_DONE = False
if SCENES_DONE and not FAILURE:
    jobs = {}
    if len(read_csv(f"{OUT_ROOT}/harness/base/per_episode_repeat_state{STATE_B}.csv")) < n_ids(HARNESS_BASE_IDS):
        jobs["harness_base"] = (search_cmd(TASK_B, STATE_B, HARNESS_BASE_IDS, f"{OUT_ROOT}/harness/base"), 0)
    if len(read_csv(f"{OUT_ROOT}/harness/N0/per_episode_lora.csv")) < n_ids(HARNESS_N_SEEDS):
        jobs["harness_N0"] = (harness_n_cmd(), 1)
    if run_stage("harness", jobs, 2):
        HARNESS_DONE = run_gate("harness")

## 12. Tests A and B: the base on 6/14 and 6/20 (400 noises each, in two parts), the base and N7_0-N7_2 on 7/38 (200 each)

Eight jobs of 200 episodes, four at a time; states that failed the scene check are skipped.

In [ ]:
def test_jobs():
    jobs = {}
    for s in STATES_A:
        if not scene_passed(TASK_A, s):
            continue
        for j, ids in enumerate(A_PARTS):
            out = f"{task_dir(TASK_A)}/search/part{j}"
            if len(read_csv(f"{out}/per_episode_repeat_state{s}.csv")) < n_ids(ids):
                jobs[f"testA_state{s}_part{j}"] = (search_cmd(TASK_A, s, ids, out), len(jobs) % 2)
    if scene_passed(TASK_B, STATE_B):
        for policy in B_POLICIES:
            out = f"{task_dir(TASK_B)}/search/{policy}"
            if len(read_csv(f"{out}/per_episode_repeat_state{STATE_B}.csv")) < n_ids(B_IDS):
                jobs[f"testB_{policy}"] = (search_cmd(TASK_B, STATE_B, B_IDS, out, policy), len(jobs) % 2)
    return jobs


TESTS_DONE = False
if HARNESS_DONE and not FAILURE:
    TESTS_DONE = run_stage("tests A and B", test_jobs(), len(STATES_A) * len(A_PARTS) + len(B_POLICIES))

## 13. Replays: every success of test A re-run once with its noise id

In [ ]:
def listed(flag):
    """The lines of an analysis listing as lists of words, or None after failing the run."""
    r = sh(f"{ANALYZE} {flag}", check=False, env=RUN_ENV)
    if r.returncode != 0:
        fail(f"the listing {flag} failed")
        return None
    return [line.split() for line in r.stdout.strip().splitlines()]


REPLAYS_DONE = False
plan = listed("--replay_jobs") if TESTS_DONE and not FAILURE else None
if plan is not None:
    jobs = {}
    for t, s, ids in plan:
        out = f"{task_dir(int(t))}/replay"
        if len(read_csv(f"{out}/per_episode_repeat_state{s}.csv")) < n_ids(ids):
            jobs[f"replay_state{s}"] = (search_cmd(int(t), int(s), ids, out), len(jobs) % 2)
    print("successes to replay:", {s: ids for _, s, ids in plan} or "none", flush=True)
    REPLAYS_DONE = run_stage("replays", jobs, len(plan))

## 14. Analysis (when every episode is done)

In [ ]:
if REPLAYS_DONE and not FAILURE:
    if sh(f"{ANALYZE} --out {OUT_ROOT}/step15_report.txt", check=False, env=RUN_ENV).returncode != 0:
        fail("the analysis failed: the episodes are in the zip")
elif not FAILURE:
    print("WORK REMAINS: add this version's output as input (keep step 10's) and Save & Run All again")

## 15. Files for the repo

In [ ]:
sh(f"cd '{OUT_ROOT}' && zip -q -r /kaggle/working/step15_results.zip . -x '*.npz' '*.safetensors' '*.pt'")
sh(f"du -h /kaggle/working/step15_results.zip; du -sh {OUT_ROOT}")
print(f"session time used: {(time.time() - T_SESSION) / 3600:.1f} h")
if FAILURE:
    raise RuntimeError(FAILURE)